# Lab 01: RNN Sequence Modeling

**Module 01** | Read `notes/01-rnns-sequence-modeling.pdf` before running this notebook.

Train a character-level RNN on a small corpus. You will build the vocabulary, unroll sequences, run a full training loop, and plot the loss curve.

Run every cell top to bottom. Code is complete, no fill-in sections. Markdown cells explain what each block does and why.


## Runtime device

PyTorch can run on your NVIDIA GPU through CUDA or fall back to CPU. GPU execution moves matrix operations off the host and typically speeds up neural network training by a large factor. The next cell detects what is available and prints the active device so you know whether to expect fast training or should keep batch sizes small.


In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"CUDA enabled, {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {props.total_memory / 1e9:.1f} GB")
else:
    print("CUDA not available, using CPU. Labs still run; training may take longer.")


Character-level language modeling is the simplest setting for sequence models: each timestep is one symbol, so you can watch the network learn local patterns without tokenizer complexity. We use a short repeating corpus so training finishes quickly on CPU or GPU while still producing a visible loss drop.

The goal is next-character prediction. Given 32 consecutive characters, the model predicts the character that follows. That framing matches how autoregressive text models are trained at scale.


The corpus is fixed below. From it we derive a vocabulary (unique characters plus an index mapping) and sliding windows of length `SEQ_LEN`. Each window becomes one training example; the target is the single character immediately after the window.

Keeping batch construction explicit makes the link between raw text and tensor batches clear before any neural network code runs.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

ROOT = Path("..").resolve()
SEQ_LEN = 32
BATCH_SIZE = 64
HIDDEN_SIZE = 128
EPOCHS = 5
LR = 1e-2

CORPUS = "sequence modeling practice text " * 50
chars = sorted(set(CORPUS))
vocab_size = len(chars)
char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for ch, i in char_to_idx.items()}

indices = torch.tensor([char_to_idx[c] for c in CORPUS], dtype=torch.long)
inputs, targets = [], []
for start in range(len(indices) - SEQ_LEN):
    inputs.append(indices[start : start + SEQ_LEN])
    targets.append(indices[start + SEQ_LEN])
inputs = torch.stack(inputs)
targets = torch.stack(targets)

dataset = TensorDataset(inputs, targets)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

print(f"Corpus length: {len(CORPUS):,} chars | Vocab: {vocab_size} | Examples: {len(dataset):,}")


`CharRNN` stacks three standard pieces: an embedding turns discrete character IDs into dense vectors; an `nn.RNN` maintains a hidden state while reading the sequence; a linear layer maps the final hidden state to logits over the vocabulary.

We use the built-in RNN cell rather than a manual loop so you can focus on data flow and training. The same pattern extends to LSTM and GRU in the next lab.


In [ ]:
class CharRNN(nn.Module):
    def __init__(self, vocab_size: int, hidden_size: int = 128):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, hidden_size)
        self.rnn = nn.RNN(hidden_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        embedded = self.embed(x)
        output, _ = self.rnn(embedded)
        logits = self.fc(output[:, -1, :])
        return logits


model = CharRNN(vocab_size, HIDDEN_SIZE).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
print(model)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")


Training minimizes cross-entropy between predicted logits and the true next character. We run five epochs and record the average loss per epoch. On GPU the loop is identical; only tensor placement changes via `.to(device)`.

Shuffling batches each epoch prevents the optimizer from memorizing example order, which matters even on small corpora.


In [ ]:
loss_history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    for batch_x, batch_y in loader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)

        optimizer.zero_grad()
        logits = model(batch_x)
        loss = criterion(logits, batch_y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * batch_x.size(0)

    avg_loss = epoch_loss / len(dataset)
    loss_history.append(avg_loss)
    print(f"Epoch {epoch:02d}/{EPOCHS}, loss {avg_loss:.4f}")


A falling loss curve confirms the RNN is fitting the local statistics of the corpus. The plot below is the primary diagnostic, if loss flatlines immediately, check learning rate, device placement, or label alignment.


In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(range(1, EPOCHS + 1), loss_history, marker="o")
plt.xlabel("Epoch")
plt.ylabel("Cross-entropy loss")
plt.title("CharRNN training loss")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
